# 01. Знакомство с данными

В этом ноутбуке разберёмся, какие данные есть о клиентах Santander, насколько они полные и какими продуктами уже пользуются клиенты. После этого посмотрим на самое важное для проекта — новые подключения продуктов.

Новой покупкой считаем только переход из `0` в месяце $t$ в `1` в следующем календарном месяце $t+1$. Так мы отделяем реальное подключение продукта от обычного факта владения им.

Файл `train_ver2.csv` загружается целиком, поэтому для выполнения ноутбука потребуется достаточно оперативной памяти. Результаты ячеек очищены: запустите ноутбук заново, чтобы получить значения для полного набора данных.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
    else:
        raise RuntimeError("Запустите ноутбук из корня проекта или каталога notebooks")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from bank_recommender.constants import (
    DATE_COLUMN,
    ID_COLUMN,
    PRODUCT_COLUMNS,
    PRODUCT_DESCRIPTIONS_RU,
)
from bank_recommender.data import build_temporal_pairs, read_snapshots

raw_data_path = Path(os.getenv("DATA_PATH", "train_ver2.csv")).expanduser()
DATA_PATH = raw_data_path if raw_data_path.is_absolute() else PROJECT_ROOT / raw_data_path
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Датасет не найден: {DATA_PATH}")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Данные: {DATA_PATH}")
print("Режим загрузки: один pandas.read_csv, все клиенты")

## 1. Что лежит в исходном файле

В `train_ver2.csv` находится **13 647 309 строк и 48 колонок**. Первые 24 колонки описывают дату и профиль клиента, остальные 24 показывают, какими банковскими продуктами он владеет в конкретном месяце.

Функция `read_snapshots(DATA_PATH)` один раз читает весь файл, приводит значения к нужным типам и оставляет только те поля, которые используются в проекте. В дальнейших расчётах участвуют все клиенты и все доступные месяцы.

In [ ]:
FULL_DATA_ROWS = 13_647_309
FULL_DATA_COLUMNS = 48
header = pd.read_csv(DATA_PATH, nrows=0)
if len(header.columns) != FULL_DATA_COLUMNS:
    raise ValueError(
        f"Ожидалось {FULL_DATA_COLUMNS} колонок, найдено {len(header.columns)}"
    )

full_dataset_facts = pd.DataFrame(
    {
        "значение": [
            FULL_DATA_ROWS,
            len(header.columns),
            len(PRODUCT_COLUMNS),
            DATA_PATH.stat().st_size / 1024**3,
        ]
    },
    index=["строк данных", "колонок", "продуктовых колонок", "размер, GiB"],
)
display(full_dataset_facts)

## 2. Загружаем и приводим данные к единому виду

При загрузке убираются лишние пробелы, дата преобразуется в формат даты, а продуктовые признаки приводятся к значениям `0` и `1`. Затем записи сортируются по клиенту и месяцу.

Если для одного клиента в одном месяце встречается несколько записей, остаётся последняя. Все последующие расчёты используют подготовленную таблицу `snapshots`.

In [ ]:
snapshots = read_snapshots(DATA_PATH)

dataset_overview = pd.DataFrame(
    {
        "значение": [
            snapshots.shape[0],
            snapshots.shape[1],
            snapshots[ID_COLUMN].nunique(),
            snapshots[DATE_COLUMN].nunique(),
            snapshots[DATE_COLUMN].min().date(),
            snapshots[DATE_COLUMN].max().date(),
        ]
    },
    index=[
        "строк после нормализации",
        "колонок после нормализации",
        "уникальных клиентов",
        "месячных срезов",
        "минимальная дата",
        "максимальная дата",
    ],
)
display(dataset_overview)
display(snapshots.head(3))

## 3. Проверяем качество данных

Сначала убедимся, что после подготовки не осталось повторов для одной пары «клиент — месяц». Затем посмотрим, в каких полях чаще всего встречаются пропуски.

Пропуски в профиле клиента пока сохраняем: они будут обработаны на этапе подготовки данных перед обучением модели. Пустое значение продуктового признака считаем отсутствием продукта и заменяем на `0`.

In [ ]:
duplicate_rows = int(snapshots.duplicated().sum())
duplicate_customer_months = int(
    snapshots.duplicated([ID_COLUMN, DATE_COLUMN]).sum()
)
quality_checks = pd.Series(
    {
        "полных дубликатов после нормализации": duplicate_rows,
        "дубликатов клиент-месяц после нормализации": duplicate_customer_months,
        "строк с пропуском идентификатора": int(snapshots[ID_COLUMN].isna().sum()),
        "строк с пропуском даты": int(snapshots[DATE_COLUMN].isna().sum()),
    },
    name="количество",
)
missing_summary = (
    pd.DataFrame(
        {
            "missing_count": snapshots.isna().sum(),
            "missing_pct": snapshots.isna().mean().mul(100),
        }
    )
    .sort_values(["missing_pct", "missing_count"], ascending=False)
)

display(quality_checks.to_frame())
display(missing_summary.head(15))

In [ ]:
missing_plot = (
    missing_summary.head(12)
    .sort_values("missing_pct")
    .reset_index(names="column")
)
plt.figure(figsize=(9, 5))
sns.barplot(data=missing_plot, x="missing_pct", y="column", color="#4C72B0")
plt.title("Пропуски в полном наборе данных")
plt.xlabel("Доля пропусков, %")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

## 4. Смотрим на профиль клиентов

Оценим распределение возраста, клиентского стажа и дохода. Для категориальных признаков выведем несколько самых частых значений, чтобы таблицы оставались компактными и понятными.

На этом этапе ничего не заполняем средними или медианами. Такие значения должны рассчитываться только на обучающей части данных, иначе в модель может попасть информация из будущего.

In [ ]:
numeric_profile = snapshots[["age", "antiguedad", "renta"]].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T
category_rows = []
for column in ["sexo", "segmento", "ind_actividad_cliente", "pais_residencia"]:
    shares = snapshots[column].astype("string").fillna("<NA>").value_counts(
        normalize=True
    ).head(5)
    category_rows.extend(
        {"признак": column, "значение": value, "доля": share}
        for value, share in shares.items()
    )

display(numeric_profile)
display(pd.DataFrame(category_rows))

In [ ]:
valid_age = pd.to_numeric(snapshots["age"], errors="coerce")
valid_age = valid_age[valid_age.between(18, 100)]
plt.figure(figsize=(9, 4.5))
sns.histplot(valid_age, bins=40, color="#55A868")
plt.title("Распределение возраста после ограничения 18–100 лет")
plt.xlabel("Возраст")
plt.ylabel("Количество снимков")
plt.tight_layout()
plt.show()

## 5. Какие продукты уже есть у клиентов

Продуктовые признаки в текущем месяце показывают существующий портфель клиента. Это полезная информация для модели, но не целевая переменная.

Также эти частоты помогают увидеть самые массовые и самые редкие продукты. Из-за большого числа нулей обычная доля правильных ответов (accuracy) здесь была бы малоинформативна.

In [ ]:
current_product_table = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "owners_in_dataset": [int(snapshots[p].sum()) for p in PRODUCT_COLUMNS],
        "snapshot_prevalence": [float(snapshots[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("snapshot_prevalence", ascending=False, ignore_index=True)
display(current_product_table.head(12))

In [ ]:
product_plot = current_product_table.head(12).sort_values("snapshot_prevalence")
plt.figure(figsize=(10, 6))
sns.barplot(
    data=product_plot,
    x="snapshot_prevalence",
    y="description",
    color="#C44E52",
)
plt.title("Наиболее распространённые текущие продукты")
plt.xlabel("Доля клиентских снимков")
plt.ylabel("Продукт")
plt.tight_layout()
plt.show()

## 6. Находим новые покупки

Функция `build_temporal_pairs` сопоставляет запись клиента только со следующим календарным месяцем. Для каждого продукта рассчитывается

$$y=\max(x_{t+1}-x_t, 0).$$

Переход `0→1` означает новую покупку. Переход `1→0` означает отказ от продукта и не считается положительным событием. Если между записями пропущен месяц, такая пара не создаётся. Признаки всегда берутся из месяца $t$, поэтому модель не видит будущие данные.

In [ ]:
pairs = build_temporal_pairs(snapshots)
additions_per_pair = pairs.targets[PRODUCT_COLUMNS].sum(axis=1)
target_month = (pairs.periods + 1).astype(str)

pair_summary = pd.Series(
    {
        "соседних пар клиент-месяц": len(pairs.features),
        "всего событий 0→1": int(additions_per_pair.sum()),
        "доля пар хотя бы с одной покупкой": float((additions_per_pair > 0).mean()),
        "среднее число покупок на пару": float(additions_per_pair.mean()),
        "максимум покупок в одной паре": int(additions_per_pair.max()),
    },
    name="значение",
)
addition_table = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "addition_events": [int(pairs.targets[p].sum()) for p in PRODUCT_COLUMNS],
        "addition_rate": [float(pairs.targets[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("addition_events", ascending=False, ignore_index=True)

display(pair_summary.to_frame())
display(addition_table.head(12))

In [ ]:
target_frame = pairs.targets[PRODUCT_COLUMNS].copy()
target_frame["target_month"] = target_month.to_numpy()
monthly_product_additions = target_frame.groupby("target_month")[PRODUCT_COLUMNS].sum()
monthly_pair_count = target_frame.groupby("target_month").size()
monthly_additions = pd.DataFrame(
    {
        "addition_events": monthly_product_additions.sum(axis=1),
        "customer_pairs": monthly_pair_count,
    }
)
monthly_additions["events_per_1000_pairs"] = (
    1000 * monthly_additions["addition_events"] / monthly_additions["customer_pairs"]
)
display(monthly_additions)

plt.figure(figsize=(10, 4.5))
sns.lineplot(
    data=monthly_additions.reset_index(),
    x="target_month",
    y="events_per_1000_pairs",
    marker="o",
    color="#8172B2",
)
plt.title("Динамика новых покупок по целевому месяцу")
plt.xlabel("Месяц покупки (t+1)")
plt.ylabel("Событий 0→1 на 1000 пар")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Что узнали из анализа

1. Новые подключения встречаются редко, особенно для отдельных продуктов. Поэтому основная метрика проекта — MAP@7, а дополнительно используются Recall@7, Precision@7, покрытие каталога и macro PR-AUC.
2. Текущий портфель и новая покупка — разные вещи. Модель учится именно на переходах `0→1`, а уже подключённые продукты исключаются из рекомендаций.
3. В профиле клиентов есть пропуски и редкие категории. Правила их обработки нужно подбирать только по обучающей части данных.
4. Данные зависят от времени, поэтому случайное разделение строк не подходит. Для проверки модели используются апрельские признаки и покупки мая 2016 года, а более ранние месяцы идут в обучение.
5. Последний доступный снимок относится к маю 2016 года. После выбора модели её можно переобучить на всей доступной истории и сформировать рекомендации на июнь.
6. Весь набор загружается и обрабатывается за один запуск. Это упрощает логику подготовки данных, но требует достаточно оперативной памяти и времени.